In [10]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

# =========================================================================
# ─── CONFIGURATION DE L'EXPÉRIENCE COLAB (FEDAF) ──────────────────────────
# =========================================================================
PATH_GLOBAL_CHECKPOINT = "global_checkpoint.pth"
PATH_FEDERATED_SCALER  = "federated_scaler.pkl"
PATH_CLIENT_TRAIN_CSV  = "/content/client_1_multiclass_train.csv"  # Contient les 40000 Generic
PATH_CLIENT_TEST_CSV   = "/content/global_test_multiclass.csv"   # Ton fichier de validation complet

NUM_CLASSES = 10
BATCH_SIZE = 512
GLOBAL_ATTACK_NAMES = [
    'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers',
    'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms'
]
GENERIC_INDEX = GLOBAL_ATTACK_NAMES.index('Generic')

# =========================================================================
# ─── ARCHITECTURE DU MODÈLE IDS ( MLP STRICTEMENT IDENTIQUE ) ────────
# =========================================================================
class IDS_MLP(nn.Module):
    def __init__(self, input_size, num_classes=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.net(x)

# =========================================================================
# ─── EXTRACTEUR ET ALIGNEMENT DES DONNÉES (SANS MQTT) ────────────────────
# =========================================================================
def prepare_colab_data():
    print("📜 [COLAB] Lecture des fichiers de données brutes...")
    train = pd.read_csv(PATH_CLIENT_TRAIN_CSV)
    test = pd.read_csv(PATH_CLIENT_TEST_CSV)

    possible_target_cols = ['attack_cat', 'type', 'target', 'label']
    target_col = next((col for col in possible_target_cols if col in train.columns), train.columns[-1])

    # Drop rows where the target column has NaN values to prevent IntCastingNaNError
    train = train.dropna(subset=[target_col])
    test = test.dropna(subset=[target_col])

    le = LabelEncoder().fit(GLOBAL_ATTACK_NAMES)

    if train[target_col].astype(str).str.isnumeric().all():
        y_tr = train[target_col].astype(np.int64).values
        y_te = test[target_col].astype(np.int64).values
    else:
        y_tr = le.transform(train[target_col].astype(str).str.strip()).astype(np.int64)
        y_te = le.transform(test[target_col].astype(str).str.strip()).astype(np.int64)

    drop_cols = ['id', 'label', 'target', 'ts', 'attack_cat', 'type']
    X_tr_df = train.drop(columns=[c for c in drop_cols if c in train.columns])
    X_te_df = test.drop(columns=[c for c in drop_cols if c in test.columns])

    for col in ['proto', 'service', 'state']:
        if col in X_tr_df.columns:
            le_feat = LabelEncoder()
            all_v = pd.concat([X_tr_df[col], X_te_df[col]]).astype(str)
            le_feat.fit(all_v)
            X_tr_df[col] = le_feat.transform(X_tr_df[col].astype(str))
            X_te_df[col] = le_feat.transform(X_te_df[col].astype(str))

    with open(PATH_FEDERATED_SCALER, "rb") as f:
        global_scaler = pickle.load(f)

    X_tr = (X_tr_df.values.astype(np.float32) - global_scaler["mean"]) / global_scaler["std"]
    X_te = (X_te_df.values.astype(np.float32) - global_scaler["mean"]) / global_scaler["std"]

    return X_tr, y_tr, X_te, y_te

# =========================================================================
# ─── FONCTION D'ÉVALUATION ET DE RAPPORT ─────────────────────────────────
# =========================================================================
def evaluate_model(model, loader, message=""):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X, y in loader:
            preds = torch.argmax(model(X), dim=1)
            all_preds.extend(preds.numpy())
            all_targets.extend(y.numpy())

    all_preds, all_targets = np.array(all_preds), np.array(all_targets)
    unique = sorted(np.unique(np.concatenate([all_targets, all_preds])).tolist())
    names = [GLOBAL_ATTACK_NAMES[i] for i in unique]

    print(f"\n📊 {message}")
    print(classification_report(all_targets, all_preds, labels=unique, target_names=names, zero_division=0))

# =========================================================================
# ─── PIPELINE PRINCIPALE : SIMULATION LOCAL FEDAF ────────────────────────
# =========================================================================
def run_fedaf_experiment():
    if not os.path.exists(PATH_GLOBAL_CHECKPOINT) or not os.path.exists(PATH_FEDERATED_SCALER):
        print("❌ Erreur : Assure-toi d'avoir téléversé les fichiers requis.")
        return

    # 1. Chargement et partitionnement des données
    X_tr, y_tr, X_te, y_te = prepare_colab_data()

    # Séparation locale pour l'oubli actif et la préservation EWC
    is_generic = (y_tr == GENERIC_INDEX)
    X_unlearn = X_tr[is_generic]
    y_unlearn = y_tr[is_generic]

    X_retain = X_tr[~is_generic]
    y_retain = y_tr[~is_generic]

    print(f"\n🔹 Échantillons à FLOUTER/OUBLIER (Generic)   : {len(y_unlearn)}")
    print(f"🔹 Échantillons locaux de RÉTENTION (Sains)    : {len(y_retain)}")
    print(f"🔹 Échantillons globaux de TEST pour validation: {len(y_te)}")

    loader_unlearn = DataLoader(TensorDataset(torch.tensor(X_unlearn), torch.tensor(y_unlearn)), batch_size=BATCH_SIZE, shuffle=True)
    loader_retain  = DataLoader(TensorDataset(torch.tensor(X_retain), torch.tensor(y_retain)), batch_size=BATCH_SIZE, shuffle=True)
    loader_test    = DataLoader(TensorDataset(torch.tensor(X_te), torch.tensor(y_te)), batch_size=BATCH_SIZE, shuffle=False)

    # 2. Initialisation des modèles (Student et Teacher figé)
    student_model = IDS_MLP(input_size=X_tr.shape[1], num_classes=NUM_CLASSES)
    teacher_model = IDS_MLP(input_size=X_tr.shape[1], num_classes=NUM_CLASSES)

    checkpoint = torch.load(PATH_GLOBAL_CHECKPOINT)
    state_dict = checkpoint["state_dict"] if "state_dict" in checkpoint else checkpoint

    student_model.load_state_dict(state_dict)
    teacher_model.load_state_dict(state_dict)

    teacher_model.eval() # Le prof reste immuable en mode eval
    for param in teacher_model.parameters():
        param.requires_grad = False

    # 3. Évaluation Initiale (Baseline)
    print("\n" + "="*60 + "\n🔎 ÉVALUATION INITIALE (MODÈLE ISSU DU FL CONVERGENT)\n" + "="*60)
    evaluate_model(student_model, loader_test, message="Rapport Avant application de FedAF")

    # =========================================================================
    # 🛠️ ÉTAPE A : CALCUL DES POIDS ÉLASTIQUES (EWC FISHER MEMORY)
    # =========================================================================
    print("\n🔒 Calcul du bouclier élastique EWC sur les données saines locales...")
    student_model.eval()
    ewc_fisher = {name: torch.zeros_like(param) for name, param in student_model.named_parameters() if param.requires_grad}
    criterion_ewc = nn.CrossEntropyLoss()

    for X_b, y_b in loader_retain:
        student_model.zero_grad()
        out = student_model(X_b)
        loss = criterion_ewc(out, y_b)
        loss.backward()

        with torch.no_grad():
            for name, param in student_model.named_parameters():
                if param.requires_grad and param.grad is not None:
                    ewc_fisher[name] += (param.grad.data ** 2) * X_b.size(0)

    for name in ewc_fisher:
        ewc_fisher[name] /= len(X_retain)

    # Sauvegarde des ancrages d'origine (Poids optimaux pré-oubli)
    anchor_weights = {name: param.clone().detach() for name, param in student_model.named_parameters() if param.requires_grad}

    # =========================================================================
    # 🚀 ÉTAPE B : ENTRAÎNEMENT PAR DISTILLATION FEDAF (ACTIVE FORGETTING)
    # =========================================================================
    # Hyperparamètres de régulation FedAF
    # Hyperparamètres de régulation FedAF ajustés pour ton volume de données
    EPOCHS_FA = 4        # On passe à 4 époques pour donner le temps au floutage d'agir
    LR_FA     = 5e-5       # On augmente un tout petit peu le learning rate
    GAMMA_EWC = 15.0       # On baisse le bouclier (de 50.0 à 15.0) pour laisser le modèle respirer
    T         = 1.5        # On baisse la température pour rendre la distillation plus directive  # Température de distillation (Softmax smoothing)

    optimizer = optim.AdamW(student_model.parameters(), lr=LR_FA)
    student_model.train()

    print(f"\n🧠 Lancement de l'Oubli Actif FedAF (Distillation + EWC) pendant {EPOCHS_FA} époques...")
    for epoch in range(EPOCHS_FA):
        iter_unlearn = iter(loader_unlearn)
        epoch_loss = 0.0

        for X_ret, y_ret in loader_retain:
            optimizer.zero_grad()

            # --- 1. Distillation des classes saines (Imitation du Teacher) ---
            out_student_ret = student_model(X_ret)
            with torch.no_grad():
                out_teacher_ret = teacher_model(X_ret)

            loss_distill_retain = F.kl_div(
                F.log_softmax(out_student_ret / T, dim=1),
                F.softmax(out_teacher_ret / T, dim=1),
                reduction='batchmean'
            ) * (T ** 2)

            # --- 2. Floutage Actif de la classe cible (Generic) ---
            try:
                X_unl, _ = next(iter_unlearn)
            except StopIteration:
                iter_unlearn = iter(loader_unlearn)
                X_unl, _ = next(iter_unlearn)

            out_student_unl = student_model(X_unl)

            # Création d'une distribution uniforme neutre (10% partout) : le floutage
            uniform_target = torch.full((X_unl.size(0), NUM_CLASSES), 1.0 / NUM_CLASSES)

            loss_distill_unlearn = F.kl_div(
                F.log_softmax(out_student_unl / T, dim=1),
                uniform_target,
                reduction='batchmean'
            ) * (T ** 2)

            # --- 3. Contrainte Élastique EWC (Le bouclier des innocents) ---
            loss_ewc = 0.0
            for name, param in student_model.named_parameters():
                if param.requires_grad:
                    loss_ewc += torch.sum(ewc_fisher[name] * ((param - anchor_weights[name]) ** 2))

            # Perte Globale de FedAF
            total_loss = loss_distill_retain + loss_distill_unlearn + (GAMMA_EWC / 2.0) * loss_ewc

            total_loss.backward()
            nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=0.5)
            optimizer.step()
            epoch_loss += total_loss.item()

        print(f"   [FedAF Époque {epoch+1}/{EPOCHS_FA}] Perte combinée : {epoch_loss/len(loader_retain):.5f}")

    # 5. Évaluation Finale
    print("\n" + "="*60 + "\n📉 ÉVALUATION FINALE (MODÈLE MODIFIÉ PAR FEDAF)\n" + "="*60)
    evaluate_model(student_model, loader_test, message="Rapport Après exécution de FedAF (Modèle Désappris)")

if __name__ == "__main__":
    run_fedaf_experiment()

📜 [COLAB] Lecture des fichiers de données brutes...

🔹 Échantillons à FLOUTER/OUBLIER (Generic)   : 40000
🔹 Échantillons locaux de RÉTENTION (Sains)    : 20542
🔹 Échantillons globaux de TEST pour validation: 32313

🔎 ÉVALUATION INITIALE (MODÈLE ISSU DU FL CONVERGENT)

📊 Rapport Avant application de FedAF
                precision    recall  f1-score   support

      Analysis       0.01      0.03      0.01       151
      Backdoor       0.01      0.20      0.03        49
           DoS       0.42      0.25      0.32      2922
      Exploits       0.86      0.39      0.54      5723
       Fuzzers       0.52      0.10      0.17      1011
       Generic       0.99      0.99      0.99     11349
        Normal       0.92      0.99      0.95      9196
Reconnaissance       0.14      0.19      0.16      1737
     Shellcode       0.06      0.70      0.11       155
         Worms       0.02      0.70      0.04        20

      accuracy                           0.74     32313
     macro avg      